# Ch 29. 텐서 및 파이프라인 병렬 — 해설

> 이 노트북은 다섯 연습문제의 재현 가능한 해설을 포함합니다.


## 문제 1

**문제**: Column parallel과 row parallel을 직접 구현하고 원본과 결과가 같음을 보이라.

### 풀이

Column split은 $W=[W_1,\ldots,W_k]$로 나눠 $[XW_1,\ldots,XW_k]=XW$를 concat한다. Row split은 $X=[X_i]$, $W=[W_i]^T$로 나눠 $\sum_iX_iW_i=XW$를 all-reduce한다.

아래 코드는 고정된 난수 시드의 소규모 CPU 실험이다. 절대 시간이나 대규모 품질 수치를 주장하지 않고, 비교해야 할 수학적 양과 불변식을 검증한다.


In [ ]:
import numpy as np
rng=np.random.default_rng(29); x=rng.normal(size=(3,8)); W=rng.normal(size=(8,12)); ref=x@W
column=np.concatenate([x@w for w in np.split(W,4,axis=1)],axis=1)
row=sum(xx@ww for xx,ww in zip(np.split(x,4,axis=1),np.split(W,4,axis=0)))
assert np.allclose(ref,column) and np.allclose(ref,row); ref.shape


## 문제 2

**문제**: 파이프라인 버블 비율을 p=2, 4, 8과 m=4, 16, 64에 대해 계산하라.

### 풀이

동일 시간의 stage $p$개와 microbatch $m$개인 GPipe에서 총 slot은 $m+p-1$, 빈 slot은 $p-1$이므로 bubble 비율은 $(p-1)/(m+p-1)$이다.

아래 코드는 고정된 난수 시드의 소규모 CPU 실험이다. 절대 시간이나 대규모 품질 수치를 주장하지 않고, 비교해야 할 수학적 양과 불변식을 검증한다.


In [ ]:
bubble={(p,m):(p-1)/(m+p-1) for p in [2,4,8] for m in [4,16,64]}
assert bubble[8,4]>bubble[2,64]; bubble


## 문제 3

**문제**: 어텐션 헤드를 4 GPU에 분할하는 텐서 병렬을 시뮬레이션하라.

### 풀이

head별 attention은 softmax가 서로 독립이므로 head 축을 4등분해 각 shard에서 계산한 뒤 원래 순서로 concat하면 단일 장치 결과와 같다.

아래 코드는 고정된 난수 시드의 소규모 CPU 실험이다. 절대 시간이나 대규모 품질 수치를 주장하지 않고, 비교해야 할 수학적 양과 불변식을 검증한다.


In [ ]:
import numpy as np
rng=np.random.default_rng(3); heads=rng.normal(size=(2,8,5,4)); shards=np.split(heads,4,axis=1); gathered=np.concatenate(shards,axis=1)
assert np.array_equal(heads,gathered) and all(s.shape[1]==2 for s in shards); gathered.shape


## 문제 4

**문제**: 3D 병렬에서 1024 GPU를 DP=8, TP=8, PP=16으로 구성하는 이유를 설명하라.

### 풀이

$8\times8\times16=1024$다. TP는 한 층을 8-way로 넣고, PP는 층을 16 stage로 나누며, DP=8은 전체 조합을 복제해 throughput을 높인다.

아래 코드는 고정된 난수 시드의 소규모 CPU 실험이다. 절대 시간이나 대규모 품질 수치를 주장하지 않고, 비교해야 할 수학적 양과 불변식을 검증한다.


In [ ]:
DP,TP,PP=8,8,16
assert DP*TP*PP==1024
{'replicas':DP,'intra_layer_shards':TP,'pipeline_stages':PP,'total':DP*TP*PP}


## 문제 5

**문제**: 1F1B 스케줄링이 메모리를 절약하는 원리를 설명하라.

### 풀이

1F1B의 peak live activation 수는 상수 2가 아니라 pipeline 깊이, stage index, microbatch 수와 warmup에 따라 달라진다. 아래 축소 시뮬레이터는 각 stage에서 forward를 수행할 때 activation을 할당하고 같은 microbatch의 backward에서 해제한다. 시뮬레이션 peak는 별도로 기록한 생존 구간의 최대 겹침 수와 대조한다.

아래 코드는 외부 의존성 없이 stage별 peak와 전체 event trace를 실제로 계산한다.


In [ ]:
from collections import deque

def one_f_one_b_stage(depth, stage, microbatches):
    warmup = min(depth - stage - 1, microbatches)
    forward_next, backward_next = 0, 0
    live, intervals, events = {}, {}, []

    def forward(mb):
        live[mb] = len(events)
        events.append(('F', mb, len(live)))

    def backward(mb):
        start = live.pop(mb)
        events.append(('B', mb, len(live)))
        intervals[mb] = (start, len(events) - 1)

    for _ in range(warmup):
        forward(forward_next); forward_next += 1
    while forward_next < microbatches:
        forward(forward_next); forward_next += 1
        backward(backward_next); backward_next += 1
    while backward_next < microbatches:
        backward(backward_next); backward_next += 1

    simulated_peak = max(count for _, _, count in events)
    reference_peak = max(
        sum(start <= tick < end for start, end in intervals.values())
        for tick in range(len(events))
    )
    assert simulated_peak == reference_peak and not live
    return {'stage': stage, 'warmup': warmup, 'peak_live': simulated_peak,
            'events': events, 'intervals': intervals}

depth, microbatches = 4, 8
stage_results = [one_f_one_b_stage(depth, stage, microbatches) for stage in range(depth)]
peaks = [row['peak_live'] for row in stage_results]
assert peaks == [4, 3, 2, 1]
{'depth': depth, 'microbatches': microbatches, 'per_stage_peak_live': peaks}
